<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-08-agents-and-adk/lesson-8.7-agent-harness/notebooks/GCP_Capstone_8.7_AgentHarness.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 8.7 The Agent Harness — Three Brains Over the Kit's One `documind_tools.py`, and the Four That Ship
**Netsetos GenAI Engineering — GCP Capstone** · Module 8 · rebuilt on the live lane, 8 September 2026 · *module final*

Modules 6, 7 and 8 built the pieces: function calling, a calling loop, an MCP server, ADK agents, a LangGraph graph with memory. This lesson is where they become one shape. The tool module is the kit's, imported; three adapters wrap it and a check proves nobody copied it; the harness around each brain is spelled out (limits, context, policy, failure); one question gets three cost lines; and then the four brains the chat service ships are imported from the clone and run, and the deployed service is asked beside the API.


## Setup
7.1's, plus the chat service's URL. The idea before any code: the model does one thing; everything else is the harness.


In [ ]:
!pip install -q "langchain==1.4.0" "langgraph==1.2.11" "google-adk==2.8.0" "langchain-google-genai==4.4.0" \
                "langgraph-checkpoint-sqlite==3.1.1" "opentelemetry-sdk==1.40.0" google-genai==2.22.0 google-auth==2.57.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every agent in this module imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": f"https://documind-api-{NUMBER}.{REGION}.run.app",
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
CHAT_URL = f"https://documind-chat-{NUMBER}.{REGION}.run.app"    # the chat service (12.8): the four brains, deployed (Cell 11)
USER_ID = "priya"

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - 8.7's gate fails a paste.
# THE HARNESS IS NOT THE MODEL, and almost nothing you will debug in production is the model.
# The model does exactly one thing: given messages, produce the next message - possibly a tool
# call. Everything else is yours: how many times it may do that (a call limit); what it can see
# (context assembly, summarisation, caching); what happens when a tool fails (retry policy);
# when the MODEL fails (fallback policy); what it may never do unasked (approval gates); when
# the loop ends (stop conditions); what you can reconstruct afterwards (trajectory logging).
# That list is the harness. This lesson is where Modules 6, 7 and 8 become one shape.
print("kit:", KIT, "| API:", os.environ["RAG_API_URL"])


## Cell 1: One tool module - the kit's
Not written by this notebook. Imported, with its signature and its importers printed.


In [ ]:
import inspect, subprocess

# ONE TOOL MODULE - THE KIT'S. Not written by this notebook, not pasted: imported from the clone, the
# same file every service in deploy/ imports. This cell shows the file, the signature the whole
# course shares, and who imports it - which is the argument of the lesson in three prints.
print("file     :", documind_tools.__file__)
print("retrieve :", inspect.signature(documind_tools.retrieve))
print("cost     :", inspect.signature(documind_tools.calculate_processing_cost))
importers = subprocess.run(["grep", "-rl", "documind_tools", f"{KIT}/deploy/services"], capture_output=True, text=True).stdout.split()
print("imported by:", sorted(p.replace(f"{KIT}/deploy/", "") for p in importers))

# retrieve() returns errors as DATA, never as an exception: a tool that raises kills the turn; a
# tool that returns {"error": ...} lets the model say "retrieval is down" and stop cleanly - a
# stop condition you chose rather than a stack trace. Read the source once:
src = inspect.getsource(documind_tools.retrieve)
print("\nreturns an error dict:", '"error"' in src, "| minted per call:", "_id_token(" in src, "| the brain label travels:", "brain" in src)


## Cell 2: Three adapters, one function
Each brain wants a different wrapper. None wants a different implementation. The assertions are the point.


In [ ]:
# THREE ADAPTERS, ONE FUNCTION. Each brain wants a different WRAPPER. None of them wants a
# different IMPLEMENTATION, and the difference between those two sentences is this whole lesson.
from google.adk.tools import FunctionTool
from langchain.tools import ToolRuntime      # what the framework injects: langgraph's, re-exported
from langchain_core.tools import tool

# 1. ADK (8.1-8.4). Reads the signature and the docstring; no schema to hand-write. The tenant
#    parameter is visible to the model here and OVERWRITTEN in a before_tool_callback (Cell 5) -
#    the shipped brain's shape.
adk_tool = FunctionTool(documind_tools.retrieve)

# 2. LangChain / LangGraph (6.4, 8.5, 8.6). This one binds the tenant, which the model must never
#    choose, so it cannot be the same object - it is an adapter: two lines, one of them delegates.
#    functools.partial does not work (pydantic's validate_arguments raises on a partial), and
#    Annotated[str, InjectedToolArg] is HALF right: it hides the argument and nothing fills it.
#    What deploy/services/chat/tools.py ships is ToolRuntime: hidden from the model, and its .context
#    is the dict the caller passes to agent.invoke(..., context=...). No context -> the notebook's tenant.
@tool
def retrieve(query: str, top_k: int = 5, runtime: ToolRuntime = None) -> dict:
    """Retrieve grounded passages from DocuMind's document collection, with the lane's cited answer.

    Args:
        query: The question, in natural language.
        top_k: How many passages to return.
    """
    tenant_id = (getattr(runtime, "context", None) or {}).get("tenant_id", TENANT)
    return documind_tools.retrieve(query, tenant_id=tenant_id, top_k=top_k, brain="langchain")


@tool
def calculate_processing_cost(total_pages: int, num_documents: int = 1, processing_type: str = "standard") -> dict:
    """Estimate document processing cost in USD and INR.

    Args:
        total_pages: Total page count across all documents.
        num_documents: How many documents those pages are spread across.
        processing_type: Service tier - standard, priority, or bulk.
    """
    return documind_tools.calculate_processing_cost(total_pages, num_documents, processing_type)


# 3. MCP (7.1-7.2). A separate PROCESS - the deployed documind-mcp - so it cannot share a Python
#    object with the other two. It shares the IMPORT, which is the strongest form of the same
#    guarantee, and the kit's server says so on one line:
line = subprocess.run(["grep", "-n", "documind_tools.retrieve(", f"{KIT}/deploy/services/mcp/server.py"], capture_output=True, text=True).stdout.strip()
print("mcp server delegates:", line[:110])

# PROVE IT, because "we share the tool" is the kind of thing everyone believes and nobody checks.
assert adk_tool.func is documind_tools.retrieve, "ADK wrapped a copy"               # identity, not a lookalike
names = retrieve.func.__code__.co_names                                               # every global the body touches
assert "retrieve" in names, "the LangChain adapter does not delegate - it reimplements"
assert "post" not in names, "the LangChain adapter calls rag-api itself - that is a second copy"
print("adk        ->", adk_tool.func.__module__ + "." + adk_tool.func.__name__)
print("model sees ->", sorted(retrieve.tool_call_schema.model_json_schema()["properties"]))   # no runtime, no tenant


## Cell 3: The check that makes it a rule
A function that takes a query and reaches a corpus, not a function named retrieve. Run over the kit's tool layer, then broken on purpose.


In [ ]:
# A RULE NOBODY CHECKS IS A PREFERENCE. This is the check, and it runs in CI over the kit
# (tools/check_contract.py carries the notebook-side version: no notebook may define a retrieve()
# over HTTP). What counts as a retrieval implementation: a function that takes a QUERY and reaches
# a CORPUS - not a function named `retrieve`. Grep finds the spelling you thought of; the second
# implementation is always called something else. Parsing finds the shape.
import ast
from pathlib import Path

QUERYISH = {"query", "q", "question", "search_query"}
CORPUS = ("/v1/query", "find_nearest", "find_neighbors", "vector", "embed", "corpus", "chunk")
SCOPE = [f"{KIT}/deploy/shared/documind_tools.py", f"{KIT}/deploy/services/chat/tools.py", f"{KIT}/deploy/services/chat/brains.py",
         f"{KIT}/deploy/services/mcp/server.py", f"{KIT}/deploy/services/agent/agent.py", "."]   # the agent tool layer, and this notebook's cwd


def _delegates(node) -> bool:
    """Does this function hand off to documind_tools.retrieve? Adapters are the POINT - every brain
    needs one - and a check that flags them is a check somebody disables in week two."""
    return any(isinstance(n, ast.Call) and isinstance(n.func, ast.Attribute) and n.func.attr == "retrieve"
               for n in ast.walk(node))


def retrieval_impls(scope) -> list[str]:
    """Every PUBLIC function in scope that looks like it retrieves from the corpus ITSELF. A module's
    own private helpers (_retrieve_local, the Rs 0 lane) are part of the one implementation."""
    out = []
    files = []
    for s in scope:
        p = Path(s)
        # a file, or a directory's TOP level: the clone under /content is not scanned whole - rag-api IS an implementation
        files += [p] if p.is_file() else sorted(p.glob("*.py"))
    for p in files:
        try:
            src = p.read_text(encoding="utf-8"); tree = ast.parse(src)
        except (SyntaxError, UnicodeDecodeError, FileNotFoundError):
            continue
        for node in ast.walk(tree):
            if not isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)) or node.name.startswith("_"):
                continue
            if not ({a.arg for a in node.args.args} & QUERYISH) or _delegates(node):
                continue
            stmts = node.body[1:] if (node.body and isinstance(node.body[0], ast.Expr)
                                      and isinstance(node.body[0].value, ast.Constant)) else node.body
            body = "\n".join(ast.get_source_segment(src, s) or "" for s in stmts)   # code, not the docstring
            if any(m in body for m in CORPUS):
                out.append(f"{p.name}:{node.lineno} {node.name}()")
    return out


found = retrieval_impls(SCOPE)
print("retrieval implementations:", found)
assert found == ["documind_tools.py:%d retrieve()" % documind_tools.retrieve.__code__.co_firstlineno], found
print("PASS - one implementation, and it is the kit's")

# NOW BREAK IT ON PURPOSE. A check you have never seen fail is a check you have never tested.
Path("helpful_shortcut.py").write_text(
    "import requests\n"
    "def quick_lookup(q, tenant):\n"
    "    return requests.post('http://rag-api/v1/query', json={'query': q}).json()\n")
found = retrieval_impls(SCOPE)
print("\nafter someone adds a 'quick' helper:", found)
try:
    assert len(found) == 1
except AssertionError:
    print("CI FAILS, correctly: two implementations")
Path("helpful_shortcut.py").unlink()      # tidy up so the rest of the notebook starts clean


## Cell 4: What actually stops the loop
Four kinds of stop condition, one of them the model's. `run_limit=1` is how you test a limit.


In [ ]:
# WHAT ACTUALLY STOPS AN AGENT LOOP. There are only four kinds, and only one of them is the
# model's decision:
#   1. the model stops asking for tools           - the ONLY one you do not control
#   2. a budget is exhausted (model calls, tool calls, tokens, wall clock)
#   3. a policy refuses (approval declined, guard blocked, PII detected)
#   4. something raised and nothing caught it     - not a stop condition, an outage
# An agent with only #1 is not a system. It is a loop with a hope in it.
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware
from langchain_google_genai import ChatGoogleGenerativeAI

MODEL = "gemini-3.6-flash"
llm = ChatGoogleGenerativeAI(model=MODEL, vertexai=True, project=PROJECT_ID, location="global", thinking_level="low")
TOOLS = [retrieve, calculate_processing_cost]          # the adapters from Cell 2 - no second definition
QUESTION = "What is the notice period for a confirmed E3, and what would 250 pages cost to process?"   # lk-06 + the cost tool

# run_limit=1 is not a contrived number. A question that needs a tool needs at least TWO model
# calls: one to ask for the tool, one to read the result and answer. Cap it at one and the agent
# must stop before it can answer - deterministically, on any model. That is how you test a limit.
capped = create_agent(llm, TOOLS, middleware=[ModelCallLimitMiddleware(run_limit=1)])
out = capped.invoke({"messages": [{"role": "user", "content": QUESTION}]})
print("with run_limit=1 :", out["messages"][-1].content[:110])

normal = create_agent(llm, TOOLS, middleware=[ModelCallLimitMiddleware(run_limit=10)])
out = normal.invoke({"messages": [{"role": "user", "content": QUESTION}]})
print("with run_limit=10:", out["messages"][-1].content[:110])
assert any(getattr(m, "tool_calls", None) for m in out["messages"]), "no tool call under the normal budget"
# exit_behavior decides WHO finds out: 'end' returns a polite message and a 200; 'error' raises.


## Cell 5: The same harness in ADK
The shipped `AdkBrain`, cell for line: the guard binds the tenant, compaction and caching on the App, `max_llm_calls` with its default.


In [ ]:
# THE SAME HARNESS, EXPRESSED IN ADK - and it is deploy/services/chat/brains.py's AdkBrain, cell for line.
from google.adk.agents import LlmAgent
from google.adk.agents.run_config import RunConfig
from google.adk.agents.context_cache_config import ContextCacheConfig
from google.adk.apps import App
from google.adk.apps.app import EventsCompactionConfig

BLOCKED = {"delete_document", "send_email", "modify_access"}


def guard_tool(tool, args, tool_context):
    """before_tool_callback: return a dict to REPLACE the call, or None to let it through.
    The same seam as 6.4's wrap_tool_call and 8.6's confirm node - decide here, act after."""
    if tool.name in BLOCKED:
        return {"error": f"{tool.name} is not reachable from a model turn"}
    if tool.name == "retrieve":
        args["tenant_id"] = TENANT            # never the model's choice - the shipped brain writes the roster's answer here
        args["brain"] = "adk"
        args.pop("assertion", None)
    return None


root_agent = LlmAgent(name="documind_adk", model=MODEL,
                      instruction="Answer from DocuMind's corpus. Cite the source of every claim.",
                      tools=[adk_tool, FunctionTool(documind_tools.calculate_processing_cost)],
                      before_tool_callback=guard_tool)

app = App(
    name="documind", root_agent=root_agent,
    # Compaction is ADK's answer to 8.5's summarise node: fold old events into a summary and keep a
    # window of raw ones. Caching is billed on the PREVIOUS request's token count, and Gemini 3's
    # floor of 4,096 applies on top of min_tokens; nothing is cached on a session's first request.
    events_compaction_config=EventsCompactionConfig(compaction_interval=20, overlap_size=3),
    context_cache_config=ContextCacheConfig(min_tokens=4096, ttl_seconds=1800, cache_intervals=10),
)

# max_llm_calls IS the ADK equivalent of ModelCallLimitMiddleware. It already has a default - and
# that default is the thing to know, because a runaway stops at 500 calls, not never.
print("default max_llm_calls:", RunConfig().max_llm_calls)
run_config = RunConfig(max_llm_calls=12)
print("ours:", run_config.max_llm_calls, "| the chat service reads ADK_MAX_LLM_CALLS for the same number")


## Cell 6: The middleware stack, in order


In [ ]:
# THE LANGCHAIN HARNESS: one list, and the order in it is a design decision.
from langchain.agents.middleware import (
    SummarizationMiddleware, ContextEditingMiddleware, ClearToolUsesEdit,
    ModelCallLimitMiddleware, ToolCallLimitMiddleware, ToolRetryMiddleware,
    HumanInTheLoopMiddleware,
)

STACK = [
    # 1. Budgets first. Everything below is wasted work if the run is already over.
    ModelCallLimitMiddleware(run_limit=10, thread_limit=50, exit_behavior="end"),
    ToolCallLimitMiddleware(tool_name="retrieve", run_limit=5),
    # 2. Then make the context fit. trigger is a TUPLE - ('messages', N), ('tokens', N) or ('fraction', f).
    SummarizationMiddleware(model=llm, trigger=("messages", 20)),
    ContextEditingMiddleware(edits=[ClearToolUsesEdit(trigger=60_000, keep=3)]),
    # 3. Then policy: what needs a human before it happens (8.6's interrupt, as config).
    HumanInTheLoopMiddleware(interrupt_on={"delete_document": True}),
    # 4. Failure handling last, closest to the call it wraps.
    ToolRetryMiddleware(max_retries=2, backoff_factor=2.0, on_failure="continue"),
]

agent = create_agent(llm, TOOLS, system_prompt="Answer from DocuMind's corpus. Cite the source of every claim.", middleware=STACK)
print("middleware in order:", [m.__class__.__name__ for m in STACK])

# WHY THE ORDER: middleware wraps in list order, so the first entry is the outermost. Put
# summarisation above the call limit and you pay a model call to summarise a run that was already
# over budget. Put the retry outermost and it retries the whole stack - including the approval
# gate, which means asking a human the same question three times. create_agent also takes
# checkpointer= and store=, so 8.5's durability and 8.6's memory attach here without changing a
# line of this - which is exactly how the shipped LangChainBrain is built.


## Cell 7: `PIIMiddleware` does not know where you work


In [ ]:
# PIIMiddleware EXISTS. IT DOES NOT KNOW WHERE YOU WORK.
from langchain.agents.middleware import PIIMiddleware, PIIDetectionError
from langchain.agents.middleware.pii import PIIMatch
import re

# The built-in types are exactly these five - checked against langchain 1.4.0: email, credit_card,
# ip, mac_address, url. No PAN, no Aadhaar, no Indian mobile, no GSTIN. For a DocuMind deployment
# under DPDP that list is close to the least useful five you could have been given.
PAN_RE = re.compile(r'\b[A-Z]{5}[0-9]{4}[A-Z]\b')
AADHAAR_RE = re.compile(r'\b[2-9]\d{3}[\s\-]?\d{4}[\s\-]?\d{4}\b')
MOBILE_RE = re.compile(r'(?<!\d)(?:\+91[\-\s]?)?[6-9]\d{4}[\s\-]?\d{5}(?!\d)')


def indian_ids(text: str) -> list[PIIMatch]:
    """A custom detector. PIIMatch is a TypedDict: {type, value, start, end}."""
    out: list[PIIMatch] = []
    for kind, rx in (("pan", PAN_RE), ("mobile", MOBILE_RE), ("aadhaar", AADHAAR_RE)):
        for m in rx.finditer(text):
            out.append({"type": kind, "value": m.group(), "start": m.start(), "end": m.end()})
    return out


assert [m["type"] for m in indian_ids("PAN ABCDE1234F")] == ["pan"]
assert [m["type"] for m in indian_ids("call +919876543210")][0] == "mobile"

PII_STACK = [
    PIIMiddleware("indian_id", detector=indian_ids, strategy="block",
                  apply_to_input=True, apply_to_output=True, apply_to_tool_results=True),
    PIIMiddleware("email", strategy="redact", apply_to_output=True),
]
print("policies:", [(m.pii_type, m.strategy) for m in PII_STACK])

# NOW ATTACH IT AND TRY TO GET PAST IT. A policy you configured and never tested is a policy you
# hope you have. strategy="block" raises PIIDetectionError, so the turn fails loudly.
guarded = create_agent(llm, TOOLS, middleware=PII_STACK)
try:
    guarded.invoke({"messages": [{"role": "user", "content": "My PAN is ABCDE1234F - what is the notice period for a confirmed E3?"}]})
    print("NOT BLOCKED - the gate is not wired")
except PIIDetectionError as e:
    print("blocked, correctly:", str(e)[:90])

# And the control: the same agent, the same question, no identifier in it.
ok = guarded.invoke({"messages": [{"role": "user", "content": "What is the notice period for a confirmed E3?"}]})
print("clean question  :", ok["messages"][-1].content[:70])

# STRATEGY IS A POLICY DECISION: block (raise), redact (placeholder), mask (keep the last few
# characters - still PII), hash (stable across turns). apply_to_tool_results is the one people
# miss: your input is clean and your prompt is clean, and then a retrieved chunk carries someone's
# PAN straight into the context window. 5.5 made the same argument about the ingest path.


## Cell 8: Fallback and retry are not the same thing


In [ ]:
# TWO DIFFERENT FAILURES, TWO DIFFERENT ANSWERS, AND THEY ARE NOT INTERCHANGEABLE.
from langchain.agents.middleware import ModelFallbackMiddleware

# The MODEL failed - 503, quota, a region having a bad afternoon. Try another model. PASS AN OBJECT,
# NOT A STRING: the string form goes through init_chat_model's provider registry, which for Gemini
# reaches for langchain-google-vertexai and raises ImportError if it is not installed.
fallback_llm = ChatGoogleGenerativeAI(model="gemini-3.1-flash-lite", vertexai=True, project=PROJECT_ID, location="global")
fallback = ModelFallbackMiddleware(fallback_llm)      # the FALLBACKS, in order; the primary is create_agent's model

resilient = create_agent(llm, TOOLS, middleware=[
    ModelCallLimitMiddleware(run_limit=10),
    fallback,
    ToolRetryMiddleware(max_retries=2, backoff_factor=2.0, on_failure="continue"),   # the TOOL failed: retry with backoff
])
out = resilient.invoke({"messages": [{"role": "user", "content": QUESTION}]})
print("primary:", MODEL, "-> fallback: gemini-3.1-flash-lite")
print("answered:", out["messages"][-1].content[:80])

# Notice what that proves and what it does not: the agent WORKS, so the fallback was never reached.
# WHAT NEITHER OF THESE DOES, and both get credited with:
#   - a fallback does not make a wrong answer right. It makes an UNAVAILABLE model available.
#   - a retry does not fix a bad request. Retrying a 400 three times is three 400s and six seconds.
#   - cheaper fallback means cheaper AND WORSE. Log which model answered, or you will compare
#     quality across two models and not know it.


## Cell 9: One question, three brains, three cost lines


In [ ]:
# THE POINT OF ALL OF IT: one question, three brains, one tool, three cost lines.
import asyncio, time
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as gt

PRICES = {"gemini-3.6-flash": (1.50, 7.50), "gemini-3.1-flash-lite": (0.25, 1.50), "gemini-3.1-pro-preview": (2.00, 12.00)}
USD_INR = 85


def cost_line(model, tin, tout, seconds):
    pin, pout = PRICES[model]
    usd = (tin / 1e6) * pin + (tout / 1e6) * pout
    return f"{tin:>6} in / {tout:>5} out  ${usd:.5f}  Rs {usd * USD_INR:.4f}  {seconds:.1f}s"


def usage_of(messages):
    tin = tout = 0
    for m in messages:
        u = getattr(m, "usage_metadata", None) or {}
        tin += u.get("input_tokens", 0); tout += u.get("output_tokens", 0)
    return tin, tout


Q = "What is the notice period for a confirmed E3?"        # golden row lk-06: 60 days, NP-03
results = {}

# --- brain 1: LangChain agent, the middleware harness from Cell 6 ---
t0 = time.perf_counter()
out = agent.invoke({"messages": [{"role": "user", "content": Q}]})
results["langchain"] = (*usage_of(out["messages"]), time.perf_counter() - t0)

# --- brain 2: the hand-built LangGraph graph (8.5/8.6): edges and nodes, nothing done for you ---
from typing import Literal
from langgraph.graph import StateGraph, MessagesState, START
from langgraph.prebuilt import ToolNode

graph_llm = llm.bind_tools(TOOLS)
b = StateGraph(MessagesState)
b.add_node("agent", lambda s: {"messages": [graph_llm.invoke(s["messages"])]})
b.add_node("tools", ToolNode(TOOLS))
b.add_edge(START, "agent")
b.add_conditional_edges("agent", lambda s: "tools" if getattr(s["messages"][-1], "tool_calls", None) else "__end__")
b.add_edge("tools", "agent")
graph_brain = b.compile()

t0 = time.perf_counter()
out = graph_brain.invoke({"messages": [{"role": "user", "content": Q}]})
results["langgraph"] = (*usage_of(out["messages"]), time.perf_counter() - t0)


# --- brain 3: the ADK agent from Cell 5 ---
async def ask_adk(question: str):
    svc = InMemorySessionService()
    sess = await svc.create_session(app_name="documind", user_id=USER_ID)
    runner = Runner(app=app, session_service=svc)
    tin = tout = 0
    async for ev in runner.run_async(user_id=USER_ID, session_id=sess.id,
                                     new_message=gt.Content(role="user", parts=[gt.Part(text=question)]), run_config=run_config):
        u = getattr(ev, "usage_metadata", None)
        if u:
            tin += u.prompt_token_count or 0; tout += u.candidates_token_count or 0
    return tin, tout


t0 = time.perf_counter()
tin, tout = await ask_adk(Q)
results["adk"] = (tin, tout, time.perf_counter() - t0)

print(f"{'brain':11} {'tokens':>22}  cost per answer")
for name, (tin, tout, secs) in results.items():
    print(f"{name:11} {cost_line(MODEL, tin, tout, secs)}")

# The tool was identical in all three. Every difference above is the HARNESS - how much history it
# sent, how many calls it took, what it cached. On a question this short the two LangChain-family
# rows are nearly the same: summarisation never triggers, the limits are never approached, so the
# elaborate stack costs what the bare graph costs. A harness is insurance, and insurance is
# invisible until the day it pays. Compare again on a forty-turn conversation and the rows separate.


## Cell 10: What ships, run
`deploy/services/chat/brains.py` imported from the clone: four brains, one tool layer, one switch.


In [ ]:
# WHAT SHIPS, RUN. The three harnesses above are ONE service with a switch: deploy/services/chat/
# brains.py - langchain, langgraph, adk, and direct (no loop: one retrieve(), the API's own grounded
# answer, the floor the other three have to beat). Import it from the clone and run all four.
import sys
sys.path.insert(0, f"{KIT}/deploy/services/chat")
import brains, tools as chat_tools
from langgraph.checkpoint.memory import InMemorySaver


def thread_config(tenant_id: str, user_id: str, session_id: str) -> dict:
    return {"configurable": {"thread_id": f"{tenant_id}:{user_id}:{session_id}"}}      # 8.5's boundary, chat/agent.py's line


context = {"tenant_id": TENANT, "user_id": USER_ID, "assertion": "", "brain": ""}       # what agent.py builds from the verified identity
print("shipped tools:", [t.name for t in chat_tools.TOOLS], "| blocked:", sorted(chat_tools.BLOCKED))
print(f"{'brain':10} {'ms':>6}  tool calls / refusals / answer")
for name in brains.BRAINS:
    brain = brains.build(name, InMemorySaver())
    t0 = time.perf_counter()
    # asyncio.to_thread: the ADK brain calls asyncio.run() for itself, which a notebook's running loop forbids;
    # the chat service is sync FastAPI and has no such loop. Same code, one wrapper here.
    out = await asyncio.to_thread(brain.answer, Q, config=thread_config(TENANT, USER_ID, f"s-{name}"), context={**context, "brain": name})
    ms = int((time.perf_counter() - t0) * 1000)
    print(f"{name:10} {ms:>6}  {out.get('tool_calls')} / {out.get('refusals')} / {str(out.get('answer'))[:60]!r}")
    assert "retrieve" in (out.get("tool_calls") or []), f"{name} answered without retrieving"


## Cell 11: The deployed service and the API, one chunk
After `make smoke-chat` passes on the lane.


In [ ]:
CHAT_DEPLOYED = False     # flip after `make smoke-chat` passes in Cloud Shell (the chat service on the lean lane)

# THE DEPLOYED CHAT SERVICE AND THE API, ONE CHUNK. /v1/chat with brain=direct returns the lane's
# citations; the kit's retrieve() asks the API directly as the same identity. The chunk ids agree,
# because there is one corpus and one retrieve() and the service never made an answer of its own.
# The three agent brains answer the same question from the same service; their trace names retrieve.
import requests

if CHAT_DEPLOYED:
    headers = {"Authorization": f"Bearer {documind_tools._id_token(CHAT_URL)}"}      # the bearer leg (12.8): no person, no assertion
    print("health:", requests.get(f"{CHAT_URL}/health", headers=headers, timeout=60).json())
    rows = {}
    for name in brains.BRAINS:
        headers = {"Authorization": f"Bearer {documind_tools._id_token(CHAT_URL)}"}
        r = requests.post(f"{CHAT_URL}/v1/chat", json={"question": Q, "session_id": f"nb-{name}", "brain": name}, headers=headers, timeout=180)
        rows[name] = r.json() if r.status_code == 200 else {"error": r.status_code, "detail": r.text[:120]}
        print(f"{name:10} {rows[name].get('latency_ms', '?'):>6} ms  tools={rows[name].get('tool_calls')}  {str(rows[name].get('answer', rows[name]))[:70]!r}")
    chat_chunks = {c["chunk_id"] for c in rows["direct"].get("citations", [])}
    api_chunks = {c["chunk_id"] for c in documind_tools.retrieve(Q, tenant_id=TENANT, top_k=5).get("citations", [])}
    print("\nchat cited:", sorted(chat_chunks)[:3], "\nAPI cited :", sorted(api_chunks)[:3])
    print("same chunks:", bool(chat_chunks & api_chunks), "- one corpus, two surfaces, one citation")
else:
    print("CHAT_DEPLOYED=False - deploy documind-chat on the lane (make build deploy-services SERVICES_lean=chat), run make smoke-chat, flip the switch")


## Cell 12: The trajectory, the endpoint, the gates


In [ ]:
# WHAT SHIPS, AND WHAT YOU CAN RECONSTRUCT AFTERWARDS. One span per turn, one child span per tool
# call: THIS is your agent's audit trail. Log the trajectory, not the answer - six weeks from now
# the question is never "what did it say" but "why did it say that", and the answer is in which
# tools ran, in what order, with what arguments, and which model produced the final message.
from opentelemetry import trace

tracer = trace.get_tracer("documind.agent")


def traced_turn(brain: str, question: str, tenant: str):
    with tracer.start_as_current_span("agent.turn") as span:
        span.set_attribute("documind.brain", brain)          # langchain | langgraph | adk | direct
        span.set_attribute("documind.tenant_id", tenant)     # NEVER the question text
        span.set_attribute("documind.model", MODEL)
        # ... run the brain, then:
        span.set_attribute("documind.tool_calls", 1)
        span.set_attribute("documind.model_calls", 2)
        span.set_attribute("documind.stop_reason", "model_finished")
    return span.get_span_context().trace_id


print("trace id:", hex(traced_turn("langgraph", Q, TENANT)))
# 0x0 without an exporter configured: the no-op tracer is what you get until Cloud Trace is wired,
# and it is worth seeing once so you recognise it when a dashboard is mysteriously empty.

# THE ENDPOINT, THIN, and the one already deployed:
#   POST /v1/chat  {question, session_id, brain}     brain = langchain | langgraph | adk | direct
#   -> {answer, tool_calls, refusals, brain, session_id, latency_ms}
# tenant_id is NOT in the request: it comes from the verified identity (12.8) - the person's IAP
# assertion through the UI, or an ID token minted for the service's url from a notebook - and the
# thread the checkpointer keys on is tenant:user:session (8.5). Every usage row names the brain.
#
# THE CI GATES, and the reason any of this holds:
#   python tools/check_contract.py      no notebook defines a retrieve() over HTTP; the lane notebooks import the kit
#   python tools/check_auth_wiring.py   the tenant reaches the tools through the runtime, never the schema; the peer imports nothing
#   make smoke-chat PROJECT=...         four brains, one question, retrieve in every trace, the outsider refused
#
# WHAT stop_reason CAN BE, and log it every time:
#   model_finished | call_limit | tool_limit | approval_declined | pii_blocked | tool_failed
# Four of those six are your harness doing its job. If you only ever see the first one in your
# dashboards, your limits are not wired - not that they are never hit.


## Module 8, finished

| Lesson | What it added | Where it runs |
|---|---|---|
| 8.1-8.2 | ADK agents and teams on the kit's tool layer | inside the kit, on the one `retrieve()` |
| 8.3 | an agent for a managed runtime, over the MCP server | outside the kit |
| 8.4 | an A2A peer on Cloud Run, over the MCP server | outside the kit, the lane's third service |
| 8.5-8.6 | a LangGraph brain that survives a restart, with memory, a gate and MCP tools | inside the kit, the chat service's second brain |
| **8.7** | **the harness, three cost lines, and the four brains that ship** | `deploy/services/chat`, deployed |

The through-line of the whole module is one sentence, and it is the same one 6.2 opened with: **the thing that decides must be separate from the thing that acts.** A router before the model, a guard before dispatch, an approval before the side effect, a limit around the loop, a roster before the corpus. Every piece of the harness is that idea wearing different clothes - and every agent in the module is either inside the kit on the one `retrieve()` or outside it on the MCP server, never a third thing.
